In [2]:
!pip install unsloth -q
!pip install --force-reinstall "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes -q

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (xformers)


In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model_name = "unsloth/qwen2.5-7b-instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="total_data.jsonl", split="train")

prompt_template = """<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>"""

def format_prompts(examples):
    instructions = examples["instruction"]
    responses    = examples["response"]
    texts = []
    for inst, resp in zip(instructions, responses):
        text = prompt_template.format(instruction=inst, response=resp)
        texts.append(text)
    return { "text" : texts }

formatted_dataset = dataset.map(format_prompts, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/444 [00:00<?, ? examples/s]

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/444 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 444 | Num Epochs = 3 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
10,1.845600
20,1.657900
30,1.531700
40,1.436500
50,1.508700
60,1.458400
70,1.353400
80,1.252600
90,1.224200
100,1.387800


wandb: WARNING URL not available in offline run


train/epoch,▁▂▂▃▃▃▄▄▅▆▆▆▇▇██
train/global_step,▁▁▂▃▃▃▄▅▅▅▆▇▇▇██
train/grad_norm,▂▁▁▁▂▄▅▅▇▄▇▇▆██
train/learning_rate,██▇▇▆▆▅▅▄▃▃▃▂▂▁
train/loss,█▆▅▅▅▅▄▃▃▄▃▂▃▂▁
total_flos,2.467925728644096e+16
train/epoch,2.68468
train/global_step,150
train/grad_norm,0.91013
train/learning_rate,0.0
train/loss,0.9719


In [7]:
model.save_pretrained("lora_nlxh_model")
tokenizer.save_pretrained("lora_nlxh_model")

model.save_pretrained_gguf("nlxh_model", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:43<00:00, 130.86s/it]


Unsloth: Merge process complete. Saved to `/content/nlxh_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['nlxh_model_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['nlxh_model_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: llama.cpp/llama-cli --model nlxh_model_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to nlxh_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f nlxh_model_gguf/Modelfile


{'save_directory': 'nlxh_model',
 'gguf_directory': 'nlxh_model_gguf',
 'gguf_files': ['nlxh_model_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'nlxh_model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [9]:
import shutil
from google.colab import files
model_path = "lora_nlxh_model"
shutil.make_archive(model_path, 'zip', model_path)
files.download(f"{model_path}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [12]:
def generate_essay(topic):
    prompt = f"""<|im_start|>user
{topic}<|im_end|>
<|im_start|>assistant b
"""
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1024,
        use_cache = True,
        temperature = 0.7,
        top_p = 0.9,
    )

    response = tokenizer.batch_decode(outputs)
    return response[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

In [16]:
test_topic = input('Nhập đề bài:')
result = generate_essay(test_topic)

print("--- BÀI LÀM CỦA AI ---")
print(result)

Nhập đề bài:"Trình bày suy nghĩ của anh/chị về tầm quan trọng của việc tự học trong thời đại công nghệ số."
--- BÀI LÀM CỦA AI ---
<|im_start|>user
"Trình bày suy nghĩ của anh/chị về tầm quan trọng của việc tự học trong thời đại công nghệ số."
<|im_start|>assistant b
ích của anh/chị về tầm quan trọng của việc tự học trong thời đại công nghệ số. Ngày nay, với sự phát triển vượt bậc của công nghệ số, việc tiếp cận thông tin trở nên dễ dàng hơn bao giờ hết. Tuy nhiên, xu hướng này cũng đặt ra câu hỏi về vai trò của việc tự học. Trong bối cảnh kiến thức được cập nhật liên tục, tự học không chỉ là một kỹ năng mà còn là một phẩm chất cần có ở mỗi cá nhân. Nó giúp chúng ta chủ động tìm kiếm, phân tích và sử dụng thông tin một cách hiệu quả, thay vì thụ động tiếp thu theo những khuôn mẫu sẵn có. Sự tự lập trong học tập rèn luyện khả năng tư duy phản biện, giúp ta không bị giới hạn bởi những quan điểm pre-made. Thêm vào đó, tự học còn thúc đẩy tinh thần sáng tạo, khả năng giải quyết vấn đề một 

In [15]:
!pip install huggingface_hub
from huggingface_hub import notebook_login

notebook_login()

In [17]:
hf_repo_id = "Hnug/qwen2.5-7b-nlxh-adapter"

model.push_to_hub(hf_repo_id)
tokenizer.push_to_hub(hf_repo_id)

print(f"Uploaded: https://huggingface.co/{hf_repo_id}")

README.md:   0%|          | 0.00/558 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 49.4kB /  162MB            

Saved model to https://huggingface.co/Hnug/qwen2.5-7b-nlxh-adapter


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9s528kuv/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

Uploaded: https://huggingface.co/Hnug/qwen2.5-7b-nlxh-adapter


In [18]:
hf_repo_id = "Hnug/qwen2.5-7b-nlxh-gguf"

from huggingface_hub import HfApi

api = HfApi()

api.create_repo(repo_id=hf_repo_id, repo_type="model", exist_ok=True)


api.upload_folder(
    folder_path="nlxh_model_gguf",
    repo_id=hf_repo_id,
    repo_type="model",
)

print(f"Uploaded: https://huggingface.co/{hf_repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5-7B-Instruct.Q4_K_M.gguf:   1%|          | 32.0MB / 4.68GB            

Uploaded: https://huggingface.co/Hnug/qwen2.5-7b-nlxh-gguf
